# Schematic: topography at coarse vs. ~50 km resolution

Two-panel orthographic-globe schematic illustrating how a coarsening grid smooths out
topographic detail -- the motivation for the topography-sensitivity experiments in this
repo. Modeled on a two-panel version of a published 4-panel schematic (coarsest and
finest resolution only), with the orthographic projection re-centered on North America
instead of the Atlantic/Europe.

The idealized-coarsening technique (build cell-bounds, conservative regrid with xesmf)
is carried over as-is from `topography_obs_models_comparisons.ipynb`, applied to the full
global ETOPO05 field instead of the North America-only subset used there. Panel A uses an
illustrative low-resolution grid (5 deg, ~550 km -- comparable to an older coarse-resolution
GCM) chosen for visual contrast; it is not one of the FLOR/HighResMIP analysis resolutions
used elsewhere in this repo. Panel B is regridded to ~50 km (0.5 deg), matching the
HighResMIP-high idealized boundary condition (`hrmip['high']`) in
`topography_obs_models_comparisons.ipynb` -- not native ETOPO05 resolution.

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

import xesmf as xe
import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
import pyproj
os.environ["PROJ_DATA"] = pyproj.datadir.get_data_dir()
os.environ.setdefault("PROJ_LIB", os.environ["PROJ_DATA"])

import cartopy
cartopy.config['data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
cartopy.config['pre_existing_data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cmocean
from cmocean import cm as cmo

%config InlineBackend.figure_format = 'retina'

# add path to custom functions
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/py_functions")
from colorbar_funcs import *

dpath0 = '/discover/nobackup/projects/giss/baldwin_nip/dmkumar'

## Load native-resolution ETOPO05 topography (global)

Same file and coordinate rename as `topography_obs_models_comparisons.ipynb`. Land and
ocean are both kept (unlike that notebook's `etopo` variable, which masks out bathymetry)
so both regridded panels show blocky ocean depth too, matching the reference schematic's
look. This native field is only the regridding source below -- neither panel plots it
directly.

In [ ]:
etopo_full = xr.open_dataset(f'{dpath0}/topo_files/obs.etopo5.zsurf.nc').ROSE
etopo_full = etopo_full.rename({'ETOPO05_Y': 'lat', 'ETOPO05_X': 'lon'})

## Idealized coarsening

Identical technique to the "INTERPOLATE OBSERVED TOPOGRAPHY TO IDEALIZED NOMINAL
RESOLUTIONS" cell in `topography_obs_models_comparisons.ipynb`: build cell-edge bounds
on the native grid and on a coarse target grid, then conservative-regrid with xesmf.
Here the source is the *global*, unmasked field, and both target grids span the same
global lat/lon range -- one coarse (5 deg, illustrative), one at ~50 km (0.5 deg,
matching the `50km` bin already used for the HighResMIP-high idealized boundary
condition in `topography_obs_models_comparisons.ipynb`).

In [ ]:
resolutions = ['coarse', 'fine']
grid_spc_deg = [5.0, 0.5]  # coarse: illustrative low-res grid (~550 km); fine: ~50 km

# --- define etopo native source grid --- #
lat_fine = etopo_full.lat
lon_fine = etopo_full.lon
dlat_fine = float(lat_fine[1] - lat_fine[0])
dlon_fine = float(lon_fine[1] - lon_fine[0])
etopo_src = etopo_full.to_dataset(name='topo')
etopo_src['lat_b'] = xr.DataArray(
    np.append(lat_fine.values - dlat_fine/2, lat_fine.values[-1] + dlat_fine/2),
    dims=['lat_b']
)
etopo_src['lon_b'] = xr.DataArray(
    np.append(lon_fine.values - dlon_fine/2, lon_fine.values[-1] + dlon_fine/2),
    dims=['lon_b']
)

# --- construct global target grids, regrid --- #
etopo_regridded = {}
for res, deg in zip(resolutions, grid_spc_deg):
    lat_coarse = np.arange(lat_fine.min(), lat_fine.max() + 0.01, deg)
    lon_coarse = np.arange(lon_fine.min(), lon_fine.max() + 0.01, deg)
    dlat = lat_coarse[1] - lat_coarse[0]
    dlon = lon_coarse[1] - lon_coarse[0]
    coarse_lat_b = np.append(lat_coarse - dlat/2, lat_coarse[-1] + dlat/2)
    coarse_lon_b = np.append(lon_coarse - dlon/2, lon_coarse[-1] + dlon/2)

    coarse_grid = xr.Dataset({
        'lat':   (['lat'],   lat_coarse),
        'lon':   (['lon'],   lon_coarse),
        'lat_b': (['lat_b'], coarse_lat_b),
        'lon_b': (['lon_b'], coarse_lon_b),
    })

    regridder = xe.Regridder(etopo_src, coarse_grid, method='conservative', periodic=True)
    etopo_regridded[res] = regridder(etopo_src['topo'], skipna=True, na_thres=0.75)

## Two-panel orthographic schematic, centered on North America

In [ ]:
# --- settings --- #
cmap = cmocean.tools.lighten(cmo.topo, 0.95)
vmin, vmax = -3500, 3500
levels = np.linspace(vmin, vmax, 43)
norm = mpl.colors.BoundaryNorm(levels, cmap.N)
trans = ccrs.PlateCarree()
proj = ccrs.Orthographic(central_longitude=-100, central_latitude=40)
letters = ['A', 'B']
titles = [f'{grid_spc_deg[0]:.0f}° (~{grid_spc_deg[0]*111:.0f} km)',
          f'{grid_spc_deg[1]:.1f}° (~{grid_spc_deg[1]*111:.0f} km)']
datasets = [etopo_regridded['coarse'], etopo_regridded['fine']]

text_kw2 = {'size': 20, 'weight': 'bold', 'color': 'k', 'ha': 'center', 'va': 'center'}

fig = plt.figure(figsize=(12, 6.5))
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.05)

for i, (dat, title) in enumerate(zip(datasets, titles)):
    ax = fig.add_subplot(gs[0, i], projection=proj)
    ax.set_global()
    cf = ax.pcolormesh(dat.lon, dat.lat, dat, cmap=cmap, norm=norm, shading='auto', transform=trans)
    ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=0.4)
    ax.coastlines(color='k', linewidth=0.6)
    ax.set_title(title, fontsize=15, fontweight='bold', pad=10)
    ax.text(0.02, 0.96, letters[i], transform=ax.transAxes, **text_kw2)

cax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
cbar = fig.colorbar(cf, cax=cax, extend='both', orientation='vertical')
cbar.set_ticks([-3000, -2000, -1000, 0, 1000, 2000, 3000])
cbar.set_ticklabels([-3, -2, -1, 0, 1, 2, 3])
cbar.set_label('Surface Elevation [km]', labelpad=15, rotation=270, size=14, ha='center')
cbar.ax.tick_params(labelsize=12)

plt.savefig('../figs/topo_resolution_schematic_globe.png', transparent=False, bbox_inches='tight', dpi=200)
plt.savefig('../figs/topo_resolution_schematic_globe.pdf', transparent=False, bbox_inches='tight')